# Colab T4 Validation Notebook

Runs the full pipeline end-to-end on a free Colab T4 GPU to confirm the magic-prefix artifact format, the `FrozenNgramOracle.from_bytes` reload, the SGD TTT branch, and the HedgeMixer-with-oracle path all work outside the local 4060 setup.

**This is sanity validation, not competition validation.** The model size is intentionally small enough to fit a T4 in <5 minutes; competition-scale 11L/512d numbers are out of scope here.

Steps:
1. Install deps, clone the repo
2. Download a single FineWeb sp1024 shard
3. Run `build_ngram_oracle.py --self-test`
4. Build a tiny oracle from the shard (in-memory, ~30s)
5. Run a 100-step toy training with TTT + Hedge + oracle
6. Verify final artifact size, BPB, and oracle roundtrip

In [ ]:
# 1. Environment
!pip install -q zstandard sentencepiece
import torch, os, sys, time
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB" if torch.cuda.is_available() else 'N/A')
print(f"PyTorch: {torch.__version__}")

In [ ]:
# 2. Clone repo + download one data shard
!git clone https://github.com/openai/parameter-golf.git /content/pgolf 2>/dev/null || (cd /content/pgolf && git pull)
%cd /content/pgolf
!python data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1

In [ ]:
# 3. Drop our submission's train_gpt.py + build_ngram_oracle.py into the repo.
# Adjust the next two cells to point at wherever you uploaded the files.
# Easiest: upload via Colab's file browser, then:
import shutil
shutil.copy('/content/train_gpt.py', '/content/pgolf/train_gpt.py')
shutil.copy('/content/build_ngram_oracle.py', '/content/pgolf/build_ngram_oracle.py')
print('Submission files copied to repo')

In [ ]:
# 4. FNV-1a self-test (proves the NumPy build hash matches the Torch lookup hash)
!python build_ngram_oracle.py --self-test

In [ ]:
# 5. Build a small oracle (5M tokens slice from the single shard for speed; ~10s)
import sys, io, struct, glob, time
sys.path.insert(0, '/content/pgolf')
import numpy as np
from build_ngram_oracle import count_order, counts_to_int8_logprobs, BUCKET_CONFIGS, VOCAB, MAGIC, USE_ZSTD
try:
    import zstandard as zstd
except ImportError:
    import zlib

shards = glob.glob('/content/pgolf/data/datasets/fineweb10B_sp1024/fineweb_train_*.bin')
assert shards, 'No shards found — did the data download succeed?'
header = np.fromfile(shards[0], dtype='<i4', count=256)
num_tokens = int(header[2])
tokens = np.fromfile(shards[0], dtype='<u2', count=min(num_tokens, 5_000_000), offset=256*4).astype(np.int32)
print(f'Loaded {len(tokens):,} tokens')

t0 = time.time()
all_counts = {order: count_order(tokens, order, buckets) for order, buckets in BUCKET_CONFIGS.items()}
print(f'Counted in {time.time()-t0:.1f}s')

buf = io.BytesIO()
buf.write(struct.pack('<II', MAGIC, len(BUCKET_CONFIGS)))
for order in sorted(BUCKET_CONFIGS):
    data = counts_to_int8_logprobs(all_counts[order])
    rows, cols = data.shape
    buf.write(struct.pack('<III', order, rows, cols))
    buf.write(data.tobytes())
raw = buf.getvalue()
if USE_ZSTD:
    compressed = zstd.ZstdCompressor(level=22).compress(raw)
else:
    compressed = zlib.compress(raw, level=9)
with open('/content/ngram_oracle.bin', 'wb') as f:
    f.write(compressed)
print(f'Oracle: {len(compressed):,} bytes ({len(compressed)/1024/1024:.2f} MB)')

In [ ]:
# 6. Run the full pipeline (training + quantize + bundle + reload + SGD TTT + HedgeMixer-with-oracle + eval)
# T4 16GB config: 6L/384d/MLP=1280, 100 steps, ~3-5 min total.
import subprocess, os

env = os.environ.copy()
env.update({
    'TORCHDYNAMO_DISABLE': '1',  # T4 lacks the Triton path used by torch.compile fullgraph
    'SEED': '42',
    'NUM_LAYERS': '6', 'MODEL_DIM': '384', 'NUM_HEADS': '6', 'NUM_KV_HEADS': '6', 'MLP_HIDDEN': '1280',
    'TRAIN_SEQ_LEN': '512', 'TRAIN_BATCH_TOKENS': '32768', 'VAL_BATCH_SIZE': '32768', 'VOCAB_SIZE': '1024',
    'ITERATIONS': '100', 'WARMDOWN_ITERS': '30', 'WARMUP_STEPS': '5',
    'MAX_WALLCLOCK_SECONDS': '120',
    'VAL_LOSS_EVERY': '50', 'TRAIN_LOG_EVERY': '20',
    'VALUE_RESIDUAL': '1', 'GATED_ATTENTION': '1', 'EMA_ENABLED': '1', 'EMA_DECAY': '0.9985',
    'LATE_QAT': '0',
    'XSA_LAYERS': '2', 'ROPE_DIMS': '16', 'LN_SCALE': '1',
    'LEAKY_RELU_SLOPE': '0.75',
    'MUON_WD': '0.04', 'ADAM_WD': '0.04',
    'MATRIX_LR': '0.025', 'SCALAR_LR': '0.025', 'TIED_EMBED_LR': '0.035',
    'MUON_MOMENTUM': '0.99', 'MUON_MOMENTUM_WARMUP_START': '0.92', 'MUON_MOMENTUM_WARMUP_STEPS': '50',
    # SGD TTT + HedgeMixer-with-oracle
    'TTT_ENABLED': '1', 'TTT_OPTIMIZER': 'sgd',
    'TTT_LR': '0.002', 'TTT_EPOCHS_PER_CHUNK': '2',
    'TTT_CHUNK_TOKENS': '16384', 'TTT_FREEZE_BLOCKS': '4', 'TTT_TIME_BUDGET': '60',
    'EVAL_STRIDE': '64',
    'BIGRAM_BUCKETS': '4096', 'BIGRAM_EMBED_DIM': '96',
    'CROWN_LAMBDA': '0.0',
    'HEDGE_ENABLED': '1', 'HEDGE_ETA': '0.01',
    'HEDGE_TRIGRAM_BUCKETS': '4096', 'HEDGE_CACHE_GAMMA': '0.999',
    'NGRAM_ORACLE_PATH': '/content/ngram_oracle.bin',
    'RUN_ID': 'colab_t4_sanity',
})

proc = subprocess.Popen(
    ['python', 'train_gpt.py'],
    env=env, cwd='/content/pgolf',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

# Stream + filter output
interesting = ('oracle:', 'model_params', 'step:', 'warmup_step', 'ema:', 'Serialized', 'ttt_score',
               'final_int6', 'peak memory', 'PGAR', 'magic', 'self-test', 'WARN', 'ttt_chunk:')
for line in proc.stdout:
    line = line.rstrip()
    if any(k in line for k in interesting):
        print(line)

proc.wait()
print(f'\n=== Exit code: {proc.returncode} ===')

In [ ]:
# 7. Verify final artifact (size + magic prefix)
import struct, os
artifact_path = '/content/pgolf/final_model.int6.ptz'
if os.path.exists(artifact_path):
    size = os.path.getsize(artifact_path)
    print(f'Artifact: {size:,} bytes ({size/1024/1024:.2f} MB)')
    print(f'Budget remaining: {(16*1024*1024 - size)/1024/1024:.2f} MB')
    with open(artifact_path, 'rb') as f:
        head = f.read(16)
    magic, version, neural_len, oracle_len = struct.unpack('<IBxxxII', head)
    expected_magic = 0x50474152  # 'PGAR'
    print(f'Magic: {magic:#x} ({"OK" if magic == expected_magic else "FAIL — expected PGAR"})')
    print(f'Version: {version}')
    print(f'Neural blob: {neural_len:,} bytes ({neural_len/1024/1024:.2f} MB)')
    print(f'Oracle blob: {oracle_len:,} bytes ({oracle_len/1024/1024:.2f} MB)')
else:
    print('No artifact produced — check the previous cell for errors')

## Expected output

On a clean T4 run, you should see:

```
[self-test] FNV-1a NumPy/Torch agree on 1000 samples
oracle:loaded for training orders=[1, 2, 3, 4, 5, 6, 7, 8] alpha=0.0
model_params:~10M
step:0/100 ... step:100/100
ema:applying EMA weights
Serialized model int6+zstd-22: neural=~5MB oracle=~3MB total=~8MB
oracle:loaded from artifact orders=[1, 2, 3, 4, 5, 6, 7, 8]
ttt_score_first:start lr=0.002 optimizer=sgd ... oracle=yes
ttt_chunk:N/N elapsed:Xs
final_int6_zstd-22_roundtrip val_bpb:~2.0
Magic: 0x50474152 (OK)
Version: 1
```

The val_bpb won't be competitive — that's not the point. The point is every component runs end-to-end on a different machine with a different GPU, with the magic-prefix artifact format intact and the oracle roundtripping correctly through `from_bytes`.